## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

## Run

In [ ]:
# TODO: Develop your agents under `agentic/agents`
# TODO: Develop your tools under `agentic/tools`
# TODO: Modify `agentic/workflow` in order to orchestrate your agents

In [10]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

# Only import the orchestrator
from agentic.workflow import orchestrator

app = orchestrator()

chat_interface(app, "MEM-DEMO-02")


Type 'exit' or 'q' to quit.

Here are the subscription details for Alice Kingsley:

- Email: alice.kingsley@wonderland.com
- Tier: basic
- Status: cancelled
- Monthly quota: 2

If you want to cancel, pause, or change the plan, tell me what you want to do and I’ll guide you.
You can renew your subscription at any time via the 'My Account' section in the CultPass app. If you need further assistance, let me know!

Sources: How to Cancel or Pause a Subscription; How to Check Your Monthly Quota; What's Included in a CultPass Subscription
(confidence=0.80)
Goodbye!


In [21]:

def extract_state(snapshot):
    """
    Robustly extract state from a LangGraph StateSnapshot across versions.

    Supports:
    - snapshot.values (dict)
    - snapshot.state (dict)
    - snapshot.checkpoint["values"] (dict)
    - iterable snapshots that contain dict writes
    - mapping-like snapshots (dict(snapshot)) if supported
    """
    if snapshot is None:
        return None

    # 1) Common attributes
    for attr in ("values", "state"):
        if hasattr(snapshot, attr):
            val = getattr(snapshot, attr)
            if isinstance(val, dict) and val:
                return val

    # 2) Some versions store values inside a checkpoint dict
    if hasattr(snapshot, "checkpoint"):
        cp = getattr(snapshot, "checkpoint")
        if isinstance(cp, dict):
            v = cp.get("values")
            if isinstance(v, dict) and v:
                return v

    # 3) Mapping-like snapshots
    try:
        as_dict = dict(snapshot)
        if isinstance(as_dict, dict) and as_dict:
            return as_dict
    except Exception:
        pass

    # 4) Iterable of writes (list/tuple of partial dict updates)
    try:
        items = list(snapshot)
        # Look from the end for the last dict-like write
        for item in reversed(items):
            if isinstance(item, dict) and item:
                return item
            # nested snapshot objects
            for attr in ("values", "state"):
                if hasattr(item, attr):
                    v = getattr(item, attr)
                    if isinstance(v, dict) and v:
                        return v
    except Exception:
        pass

    return None


In [22]:

current = app.get_state(config={"configurable": {"thread_id": "MEM-DEMO-02"}})

state = extract_state(current)
print("Extracted state is None?", state is None)
print(state)



Extracted state is None? False
{'final_answer': "You can renew your subscription at any time via the 'My Account' section in the CultPass app. If you need further assistance, let me know!\n\nSources: How to Cancel or Pause a Subscription; How to Check Your Monthly Quota; What's Included in a CultPass Subscription\n(confidence=0.80)", 'confidence': 0.7999999999999999, 'escalated': False, 'escalation_reason': None, 'next_step': 'memory', 'actions_taken': ['analysis_agent', 'memory_agent'], 'conversation_summary': 'confidence=0.7999999999999999, escalated=False'}


In [17]:


snapshots = list(
    app.get_state_history(
        config={"configurable": {"thread_id": "MEM-DEMO-02"}}
    )
)

snapshots[-1]          # not empty anymore





StateSnapshot(values=None, next=('__start__',), config={'configurable': {'thread_id': 'MEM-DEMO-02', 'checkpoint_ns': '', 'checkpoint_id': '1f14389a-f920-6b5a-bfff-a28fb9cc8510'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-04-29T05:10:06.036045+00:00', parent_config=None, tasks=(PregelTask(id='4add0d6a-75fc-4bd8-a330-f7bcce40a78c', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'ticket_id': 'MEM-DEMO-02', 'user_input': 'give subscription details about alice.kingsley@wonderland.com', 'ticket_metadata': {'urgency': 'low', 'complexity': 'low', 'status': 'open', 'issue_type': 'general'}, 'messages': [HumanMessage(content='give subscription details about alice.kingsley@wonderland.com', additional_kwargs={}, response_metadata={})]}),), interrupts=())